In [1]:
from Model import fNIRS_T
from Metrics import metrics
from Dataloader_EF import read_ef_excel_si as si_data

import torch
import os
from sklearn.metrics import precision_score, recall_score, f1_score, cohen_kappa_score
import random


class LabelSmoothing(torch.nn.Module):
	def __init__(self, smoothing=0.0):
		super(LabelSmoothing, self).__init__()
		self.confidence = 1.0 - smoothing
		self.smoothing = smoothing

	def forward(self, x, target):
		logprobs = torch.nn.functional.log_softmax(x, dim=-1)
		nll_loss = -logprobs.gather(dim=-1, index=target.unsqueeze(1))
		nll_loss = nll_loss.squeeze(1)
		smooth_loss = -logprobs.mean(dim=-1)
		loss = self.confidence * nll_loss + self.smoothing * smooth_loss
		return loss.mean()

class Trainer:
	def __init__(self, config):
		self.config = config
		self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
		self.flooding_level = config['flooding_level']

		if config['mode'] == 0 or config['mode'] == 1:
			self.model = fNIRS_T(n_class=2, sampling_point=200, dim=64, depth=6, heads=8, mlp_dim=64).to(self.device)
		elif config['mode'] == 2:
			self.model = fNIRS_T(n_class=2, sampling_point=100, dim=64, depth=6, heads=8, mlp_dim=64).to(self.device)
		self.criterion = LabelSmoothing(0.1)
		self.optimizer = torch.optim.Adam(self.model.parameters(), lr=config['learning_rate'])
		self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
			self.optimizer,
			mode='min',
			factor=0.8,
			patience=5,
			verbose=True,
			min_lr=0.1*config['learning_rate'],
		)

		os.makedirs('Results', exist_ok=True)

	def z_score_um(self, train_dataset, test_dataset, eps=1e-6):
		train_data = torch.stack([train_dataset[i][0] for i in range(len(train_dataset))])
		train_mean = train_data.mean(dim=(0,2), keepdim=True)
		train_std = train_data.std(dim=(0,2),keepdim=True)
		train_data = (train_data - train_mean) / (train_std + eps)
		train_labels = torch.stack([train_dataset[i][1] for i in range(len(train_dataset))])
		train_dataset = torch.utils.data.TensorDataset(train_data, train_labels)

		eval_data = torch.stack([test_dataset[i][0] for i in range(len(test_dataset))])
		eval_data = (eval_data - train_mean) / (train_std + eps)
		eval_labels = torch.stack([test_dataset[i][1] for i in range(len(test_dataset))])
		test_dataset = torch.utils.data.TensorDataset(eval_data, eval_labels)

		return train_dataset, test_dataset

	def train_epoch(self, epoch, train_loader):
		self.model.train()
		total_correct, total_loss = 0, 0

		for batch_nirs, batch_labels in train_loader:
			batch_nirs = batch_nirs.to(self.device, dtype=torch.float32)
			batch_labels = batch_labels.to(self.device)
			
			outputs = self.model(batch_nirs)
			loss = self.criterion(outputs, batch_labels.long())

			if epoch < 30:
				b = self.flooding_level[0]
			elif epoch < 50:
				b = self.flooding_level[1]
			else:
				b = self.flooding_level[2]

			loss = (loss - b).abs() + b
			self. optimizer.zero_grad()
			loss.backward()
			self.optimizer.step()
			
			preds = torch.argmax(outputs, dim=1)
			total_correct += (preds == batch_labels).sum().item()
			total_loss += loss.item()

		train_loss = total_loss / len(train_loader)
		train_acc = total_correct / len(train_loader.dataset)
		
		return train_loss, train_acc
	
	def evaluate_epoch(self, epoch, eval_loader):
		self.model.eval()
		total_loss, total_correct = 0, 0
		all_preds, all_labels = [], []
		
		with torch.no_grad():
			for eval_nirs, eval_labels in eval_loader:
				eval_nirs = eval_nirs.to(self.device, dtype=torch.float32)
				eval_labels = eval_labels.to(self.device)
				
				outputs = self.model(eval_nirs)
				loss = self.criterion(outputs, eval_labels.long())
				
				preds = torch.argmax(outputs, dim=1)
				total_correct += (preds == eval_labels).sum().item()
				total_loss += loss.item()
				
				all_preds.extend(preds.cpu().numpy())
				all_labels.extend(eval_labels.cpu().numpy())
		
		loss = total_loss / len(eval_loader)
		self.scheduler.step(loss)
		acc = total_correct / len(eval_loader.dataset)
		precision = precision_score(all_labels, all_preds, zero_division=0)
		recall = recall_score(all_labels, all_preds, zero_division=0)
		f1 = f1_score(all_labels, all_preds, zero_division=0)
		kappa = cohen_kappa_score(all_labels, all_preds)
		
		return loss, acc, precision, recall, f1, kappa
	
	def train_subject(self, seed, subject, mode):
		# if mode == 0:
		# 	data_root = config['mi_root'] + str(subject) + '.pkl'
		# elif mode == 1:
		# 	data_root = config['ma_root'] + str(subject) + '.pkl'
		# elif mode == 2:
		# 	data_root = config['wg_root'] + str(subject) + '.pkl'
		# with open(data_root, 'rb') as f:
		# 	data = pickle.load(f)
		
		# nirs = data['nirs']
		# labels = data['labels']
		# if mode == 0 or mode == 1:
		# 	nirs_oxy = nirs[:, :36, :]
		# 	nirs_oxy = nirs_oxy.unsqueeze(1)
		# 	nirs_deoxy = nirs[:, 36:, :]
		# 	nirs_deoxy = nirs_deoxy.unsqueeze(1)
		# 	nirs = torch.cat((nirs_oxy, nirs_deoxy), dim=1)
		# elif mode == 2:
		# 	nirs_oxy = nirs[:, :36, :]
		# 	nirs_oxy = np.expand_dims(nirs_oxy, axis=1)
		# 	nirs_deoxy = nirs[:, 36:, :]
		# 	nirs_deoxy = np.expand_dims(nirs_deoxy, axis=1)
		# 	nirs = np.concatenate((nirs_oxy, nirs_deoxy), axis=1)
		# 	nirs = torch.tensor(nirs, dtype=torch.float32)
		# 	labels = torch.tensor(labels, dtype=torch.long)
		# nirs, labels = nirs.to(self.device), labels.to(self.device)

		# train_size = int(config['ratio'] * len(labels))
		# eval_size = len(labels) - train_size
		
		# dataset = torch.utils.data.TensorDataset(nirs, labels)
		# train_dataset, eval_dataset = torch.utils.data.random_split(dataset, [train_size, eval_size])
		
		train_dataset, eval_dataset = si_data(subject, mode)
		if config['z_score']:
			train_dataset, eval_dataset = self.z_score_um(train_dataset, eval_dataset)
		train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=self.config['batch_size'], shuffle=True)
		eval_loader = torch.utils.data.DataLoader(eval_dataset, batch_size=self.config['batch_size'], shuffle=False)
		
		acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []
		
		for epoch in range(self.config['num_epochs']):
			train_loss, train_acc = self.train_epoch(epoch, train_loader)
			eval_loss, eval_acc, precision, recall, f1, kappa = self.evaluate_epoch(epoch, eval_loader)
			
			acc_list.append(eval_acc)
			precision_list.append(precision)
			recall_list.append(recall)
			f1_list.append(f1)
			kappa_list.append(kappa)
			
			print(f"Subject {subject} | Epoch {epoch+1}/{self.config['num_epochs']} | LR {self.optimizer.param_groups[0]['lr']}")
			print(f"Train Loss: {train_loss:.2f} | Train Acc: {train_acc:.2f} | Eval Loss: {eval_loss:.2f} | Eval Acc: {eval_acc:.2f}")

		metrics(subject, config['log_name'], config['log_mode'], config['metric_epoches'], seed, acc_list, precision_list, recall_list, f1_list, kappa_list)	
		return

# Hyperparameters
config = {
	'num_classes': 2,
	'batch_size': 128,
	'num_epochs': 120,
	'metric_epoches': 20,
	'learning_rate': 1e-3,
	'flooding_level': [0.45, 0.40, 0.35],
	'ratio': 0.6,
	'mode': 0,
	'log_name': '3011',
	'log_mode': 1,
	'z_score': True,
	'mi_root': '../../Dataset/EF-MI-MA/EF-PKL-MI/',
	'ma_root': '../../Dataset/EF-MI-MA/EF-PKL-MA/',
	'wg_root' : '../../Dataset/EF-WG/WG/',
}

# Initialize and run trainer
for run in range(1):
	seed = random.randint(1, 2025)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)
	mode = config['mode']
	num_subject = (5 if mode == 0 or mode == 1 else 26) # MI = 0, MA = 1, WG = 2
	for subject in range(1, num_subject+1):
		print(f"\n=== Subject {subject} ===")
		print(f'Seed is {seed}\n')
		trainer = Trainer(config)
		results = trainer.train_subject(seed, subject, mode)

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(



=== Subject 1 ===
Seed is 1852



/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


1 Training Set fNIRS torch.Size([1680, 1, 72, 200]) Labels torch.Size([1680])
1 Testing Set fNIRS torch.Size([60, 1, 72, 200]) Labels torch.Size([60])
Subject 1 | Epoch 1/120 | LR 0.001
Train Loss: 0.83 | Train Acc: 0.50 | Eval Loss: 0.70 | Eval Acc: 0.48
Subject 1 | Epoch 2/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.50 | Eval Loss: 0.70 | Eval Acc: 0.42
Subject 1 | Epoch 3/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.50 | Eval Loss: 0.71 | Eval Acc: 0.50
Subject 1 | Epoch 4/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.52 | Eval Loss: 0.70 | Eval Acc: 0.50
Subject 1 | Epoch 5/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.49 | Eval Loss: 0.70 | Eval Acc: 0.50
Subject 1 | Epoch 6/120 | LR 0.001
Train Loss: 0.69 | Train Acc: 0.52 | Eval Loss: 0.70 | Eval Acc: 0.48
Subject 1 | Epoch 7/120 | LR 0.001
Train Loss: 0.69 | Train Acc: 0.54 | Eval Loss: 0.70 | Eval Acc: 0.47
Subject 1 | Epoch 8/120 | LR 0.001
Train Loss: 0.69 | Train Acc: 0.54 | Eval Loss: 0.71 | Eval Acc: 0.50
Subject 1

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


...

3 Training Set fNIRS torch.Size([1680, 1, 72, 200]) Labels torch.Size([1680])
3 Testing Set fNIRS torch.Size([60, 1, 72, 200]) Labels torch.Size([60])
Subject 3 | Epoch 1/120 | LR 0.001
Train Loss: 0.81 | Train Acc: 0.51 | Eval Loss: 0.73 | Eval Acc: 0.45
Subject 3 | Epoch 2/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.51 | Eval Loss: 0.69 | Eval Acc: 0.52
Subject 3 | Epoch 3/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.50 | Eval Loss: 0.70 | Eval Acc: 0.50
Subject 3 | Epoch 4/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.51 | Eval Loss: 0.71 | Eval Acc: 0.50
Subject 3 | Epoch 5/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.51 | Eval Loss: 0.70 | Eval Acc: 0.50
Subject 3 | Epoch 6/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.51 | Eval Loss: 0.69 | Eval Acc: 0.50
Subject 3 | Epoch 7/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.52 | Eval Loss: 0.69 | Eval Acc: 0.57
Subject 3 | Epoch 8/120 | LR 0.001
Train Loss: 0.71 | Train Acc: 0.48 | Eval Loss: 0.69 | Eval Acc: 0.52
Subject 3

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


4 Training Set fNIRS torch.Size([1680, 1, 72, 200]) Labels torch.Size([1680])
4 Testing Set fNIRS torch.Size([60, 1, 72, 200]) Labels torch.Size([60])
Subject 4 | Epoch 1/120 | LR 0.001
Train Loss: 0.84 | Train Acc: 0.51 | Eval Loss: 0.69 | Eval Acc: 0.50
Subject 4 | Epoch 2/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.51 | Eval Loss: 0.70 | Eval Acc: 0.50
Subject 4 | Epoch 3/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.48 | Eval Loss: 0.70 | Eval Acc: 0.47
Subject 4 | Epoch 4/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.51 | Eval Loss: 0.71 | Eval Acc: 0.50
Subject 4 | Epoch 5/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.51 | Eval Loss: 0.70 | Eval Acc: 0.42
Subject 4 | Epoch 6/120 | LR 0.001
Train Loss: 0.69 | Train Acc: 0.51 | Eval Loss: 0.71 | Eval Acc: 0.50
Subject 4 | Epoch 7/120 | LR 0.0008
Train Loss: 0.70 | Train Acc: 0.51 | Eval Loss: 0.73 | Eval Acc: 0.50
Subject 4 | Epoch 8/120 | LR 0.0008
Train Loss: 0.71 | Train Acc: 0.52 | Eval Loss: 0.71 | Eval Acc: 0.50
Subject

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


5 Training Set fNIRS torch.Size([1680, 1, 72, 200]) Labels torch.Size([1680])
5 Testing Set fNIRS torch.Size([60, 1, 72, 200]) Labels torch.Size([60])
Subject 5 | Epoch 1/120 | LR 0.001
Train Loss: 0.84 | Train Acc: 0.50 | Eval Loss: 0.71 | Eval Acc: 0.50
Subject 5 | Epoch 2/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.51 | Eval Loss: 0.70 | Eval Acc: 0.47
Subject 5 | Epoch 3/120 | LR 0.001
Train Loss: 0.70 | Train Acc: 0.50 | Eval Loss: 0.70 | Eval Acc: 0.50
Subject 5 | Epoch 4/120 | LR 0.001
Train Loss: 0.69 | Train Acc: 0.52 | Eval Loss: 0.71 | Eval Acc: 0.43
Subject 5 | Epoch 5/120 | LR 0.001
Train Loss: 0.69 | Train Acc: 0.51 | Eval Loss: 0.70 | Eval Acc: 0.40
Subject 5 | Epoch 6/120 | LR 0.001
Train Loss: 0.69 | Train Acc: 0.54 | Eval Loss: 0.71 | Eval Acc: 0.38
Subject 5 | Epoch 7/120 | LR 0.001
Train Loss: 0.69 | Train Acc: 0.53 | Eval Loss: 0.71 | Eval Acc: 0.38
Subject 5 | Epoch 8/120 | LR 0.0008
Train Loss: 0.69 | Train Acc: 0.54 | Eval Loss: 0.71 | Eval Acc: 0.47
Subject 